In [ ]:
## in colab:
# %pip install transformers datasets trl

## in jupyter
# !pip install transformers datasets trl accelerate

In [5]:
# we need these libraries to do the fine-tuning

# imports the dataset
from datasets import load_dataset

# imports the transfomers fine-tuning classes and functions
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    pipeline
)
from trl import SFTTrainer, SFTConfig

In [ ]:
# loads our fine-tuning dataset: transcript text from the Love Is Blind TV show

dataset = load_dataset("gofilipa/heritage_gender")

In [ ]:
dataset

In [ ]:
dataset['train']['text'][:10]

In [ ]:
model = AutoModelForCausalLM.from_pretrained('openai-community/gpt2')
tokenizer = AutoTokenizer.from_pretrained('openai-community/gpt2')

In [11]:
pipe = pipeline('text-generation', model = model, tokenizer = tokenizer)

In [ ]:
model

In [13]:
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [14]:
# our training parameters

training_params = SFTConfig(
    output_dir= "./results",
    num_train_epochs = 1, # how many times we iterate over the dataset as a whole
    learning_rate = 2e-4, # how many "steps" we take in adjusting the parameters to make up for loss
    weight_decay = 0.001, # way of regularizing the parameters
    dataset_text_field = "text",
    report_to= "none", # this is a new param, to avoid a login to W&B
    fp16 = True,         # Enable FP16 mixed precision
    bf16 = False,        # Disable BF16 on Compute Capability < 8.0
#    use_cpu= True # only when using CPU
)

In [ ]:
# creating the "trainer" object

trainer = SFTTrainer(
    model = model,
    train_dataset = dataset['train'],
    processing_class = tokenizer,
    args = training_params
)

In [ ]:
trainer.train()

In [ ]:
trainer.save_model('./heritage')

In [ ]:
heritage = AutoModelForCausalLM.from_pretrained("./heritage")
heritage_tok = AutoTokenizer.from_pretrained("./heritage")

In [20]:
pipe = pipeline('text-generation', model=heritage, tokenizer=heritage_tok, min_length=100)

In [23]:
pipe("Gender is")

[{'generated_text': 'Gender is a social construct, and it is used to construct social institutions, policies, and institutions.”\xa0\xa0These concepts are fundamental to the American story.\xa0\xa0“The history of gender is complicated.”\xa0\xa0\xa0\xa0The American Revolution, for example, has been a tragic tale.\xa0‘The new social order is based on the idea that all males and females were equal.’\xa0\xa0“Feminism is a social construct, and it is used to construct social institutions, policies, and institutions.”'}]